# Explore MBDef

This tutorial lists (1) the ten merged terms that the most papers define and (2) every definition of **media bias**, with its source paper. It needs no machine-learning model.

MBDef has five tables ([docs/dataset/dataset.md](../docs/dataset/dataset.md)):
- a row of `instances` is an **Extracted Concept**: a term as one paper defines it;
- a row of `concepts` is a **merged term**: the Extracted Concepts whose labels are the same up to case, punctuation, spacing, plural forms, British or American spelling and word order (the Label Match rule). Merging stops there: synonyms are not merged and terms are not translated, so "frame" and "framing" stay apart;
- a row of `definitions` is one definition as one paper words it, with the references it cites;
- `papers` holds the source papers and `terms` everything the papers say about each merged term (definitions, subtypes, supertypes, synonyms, …) merged across papers.

## Load MBDef
`load()` reads MBDef's five tables from `$DATA_ROOT/MBDef/data/`, where `uv run mbdef data fetch MBDef/` puts them after checking each file's sha256. Pass a folder, `load("path/to/data")`, to read another copy of the parquet files. Each table is a [polars](https://docs.pola.rs/) DataFrame.

In [1]:
from pathlib import Path

from mbdef.analysis import load
from mbdef.analysis.queries import definitions_of

OUTPUTS = Path("outputs/dataset-basics")  # tutorials/outputs/, not tracked by git
OUTPUTS.mkdir(parents=True, exist_ok=True)

data = load()
for name, table in data.items():
    print(f"{name:12} {table.height:>7,} rows")

concepts      34,916 rows
instances     62,066 rows
definitions  103,476 rows
papers         6,115 rows
terms         34,916 rows


## The ten merged terms in the most papers
`concepts.n_papers` counts the distinct papers that define a merged term; this measures how widely the extracted literature defines it, not how often the word occurs in the papers. Ties are broken alphabetically. A paper can define a term several times, so `n_definitions` can exceed `n_papers`.

In [2]:
top_terms = (
    data["concepts"]
    .sort(["n_papers", "label", "concept_id"], descending=[True, False, False])
    .select("concept_id", "label", "n_papers", "n_instances", "n_definitions")
    .head(10)
)
top_terms.write_json(OUTPUTS / "top_terms.json")
top_terms

concept_id,label,n_papers,n_instances,n_definitions
str,str,i32,i32,i32
"""gf8d45d620d38""","""fake news""",659,659,1891
"""g4b0149fcebb5""","""misinformation""",470,474,1084
"""g7f2fbf4abd1a""","""disinformation""",433,445,1182
"""ga67b95a680c9""","""framing""",420,421,1015
"""g39d88b573c35""","""frames""",235,281,570
"""g857db6e17e11""","""hate speech""",217,218,748
"""g3655b5e0180d""","""echo chambers""",202,226,448
"""gfa9e634520e7""","""propaganda""",180,182,484
"""g79fb5907d728""","""confirmation bias""",166,167,343


## The definitions of "media bias"
`definitions_of` finds the merged term by any of its spellings (`concepts.labels`, ignoring case) and returns every definition with its paper's title, year, DOI and corpus (seed or closure). The same passage quoted by two papers stays two rows. Five are printed; the JSON file holds them all.

In [3]:
media_bias = definitions_of(data, "media bias")
print(
    f"{media_bias.height:,} definitions in {media_bias['paper_id'].n_unique():,} papers"
)
for row in media_bias.head(5).iter_rows(named=True):
    print(f"\n{row['paper_id']} | {row['title']} ({row['year']})")
    print(row["text"])
media_bias.write_json(OUTPUTS / "media_bias_definitions.json")

80 definitions in 36 papers

doi_10.1007_s00799-018-0261-y__5d654f0189b4b1c2fcb1340416338a0764c0dd7c__403f79563dc22c3c | Automated identification of media bias in news articles: an interdisciplinary literature review (2018)
the intentional, non-objective coverage of news events

doi_10.1007_s00799-018-0261-y__5d654f0189b4b1c2fcb1340416338a0764c0dd7c__403f79563dc22c3c | Automated identification of media bias in news articles: an interdisciplinary literature review (2018)
defined by the news production and consumption process

doi_10.1007_s00799-018-0261-y__5d654f0189b4b1c2fcb1340416338a0764c0dd7c__403f79563dc22c3c | Automated identification of media bias in news articles: an interdisciplinary literature review (2018)
Media bias, i.e., slanted news coverage

doi_10.1007_s00799-018-0261-y__5d654f0189b4b1c2fcb1340416338a0764c0dd7c__403f79563dc22c3c | Automated identification of media bias in news articles: an interdisciplinary literature review (2018)
media bias is a complex model of skewe

### Try another term
Change `"media bias"` to a label from `top_terms`; a label that names no merged term raises an error. The [MBDef Explorer](https://media-bias-group.github.io/MBDef/) shows the same tables in the browser. These lookups show access and provenance: how many papers define a term does not make it important, and the merged terms are not a validated taxonomy.